# Fine-tune YOLO nano on simulator frames (semantic_nav)

Run this on **Google Colab with a GPU runtime** (Runtime > Change runtime type > T4 GPU).
Training never runs on the laptop (see CLAUDE.md).

Input: a dataset recorded by `capture_frames` while the sim ran with `gt_boxes:=true`:

```
frames/
  images/<frame>.png   RGB 640x480
  labels/<frame>.txt   YOLO labels, class = COCO-80 index (from Gazebo's Label system)
  frames.csv           frame, stamp, robot pose
  dataset.yaml         (paths are for the laptop; rewritten below)
```

On the laptop: `cd ~/semantic_nav_data && zip -qr frames.zip frames`, then upload `frames.zip`
to Google Drive (or upload it directly in the cell below).

**Only do this if the M3 evaluation says detection quality is poor for classes you need.**
Fine-tuning on a few hundred frames of one simulated house will make the model better *in that
house* and usually *worse on real images* (sim-to-real gap, see the M3 report).

## 1. Environment

In [ ]:
!pip -q install ultralytics
import torch, ultralytics
print(ultralytics.__version__, 'CUDA:', torch.cuda.is_available())

## 2. Get the dataset (Drive or direct upload)

In [ ]:
import os, zipfile
USE_DRIVE = True  # False -> upload frames.zip from your computer instead
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP = '/content/drive/MyDrive/semantic_nav/frames.zip'   # adjust to where you put it
else:
    from google.colab import files
    ZIP = next(iter(files.upload()))
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/data')
ROOT = '/content/data/frames'
print(len(os.listdir(f'{ROOT}/images')), 'images,', len(os.listdir(f'{ROOT}/labels')), 'label files')

## 3. Train / val split by time blocks

Consecutive frames are near-duplicates (the robot moved 0.3 m or turned 20 deg between them).
A random per-frame split would put almost the same image in train and val and report inflated
accuracy. Splitting by **blocks of consecutive frames** keeps whole viewpoints out of training.

In [ ]:
import random, shutil, glob
frames = sorted(os.path.basename(p)[:-4] for p in glob.glob(f'{ROOT}/images/*.png'))
BLOCK = 20                       # frames per block (~one viewpoint)
blocks = [frames[i:i + BLOCK] for i in range(0, len(frames), BLOCK)]
random.Random(0).shuffle(blocks)
n_val = max(1, len(blocks) // 5)  # ~20 % validation
split = {'val': sum(blocks[:n_val], []), 'train': sum(blocks[n_val:], [])}
for part, names in split.items():
    for sub in ('images', 'labels'):
        os.makedirs(f'/content/ds/{part}/{sub}', exist_ok=True)
    for f in names:
        shutil.copy(f'{ROOT}/images/{f}.png', f'/content/ds/{part}/images/')
        if os.path.exists(f'{ROOT}/labels/{f}.txt'):
            shutil.copy(f'{ROOT}/labels/{f}.txt', f'/content/ds/{part}/labels/')
print({k: len(v) for k, v in split.items()})

## 4. Dataset YAML with all 80 COCO names

Labels use COCO indices, so we keep the 80-class head: the fine-tuned model is a drop-in
replacement (same `class_names`, same node whitelist).

In [ ]:
from ultralytics import YOLO
BASE = 'yolo11n.pt'              # or 'yolov8n.pt' -- use the model the node runs
names = YOLO(BASE).names
with open('/content/ds/data.yaml', 'w') as f:
    f.write('path: /content/ds\ntrain: train/images\nval: val/images\nnames:\n')
    f.writelines(f'  {i}: {names[i]}\n' for i in range(len(names)))
print(open('/content/ds/data.yaml').read()[:300])

## 5. Baseline: the pretrained model on the validation split

Record this number: fine-tuning is only worth it if it beats the baseline by a clear margin.

In [ ]:
base_metrics = YOLO(BASE).val(data='/content/ds/data.yaml', imgsz=640, batch=16, plots=False)
print('baseline mAP50', base_metrics.box.map50)

## 6. Fine-tune

* `freeze=10` keeps the backbone (first 10 layers) fixed: with a few hundred images, training
  only the neck + head adapts to the sim's look without destroying general features.
* Low learning rate and short schedule for the same reason.
* Default augmentation (mosaic, HSV jitter, flips, scale) stays on -- it is what makes a small
  dataset go further. Set `rect=True` only for validation-style rectangular batches.

In [ ]:
model = YOLO(BASE)
results = model.train(
    data='/content/ds/data.yaml',
    epochs=60, imgsz=640, batch=16,
    freeze=10, lr0=0.002, cos_lr=True, patience=15,
    project='/content/runs', name='semantic_nav_ft', exist_ok=True)

In [ ]:
best = YOLO('/content/runs/semantic_nav_ft/weights/best.pt')
ft_metrics = best.val(data='/content/ds/data.yaml', imgsz=640, batch=16)
print(f'mAP50 baseline {base_metrics.box.map50:.3f} -> fine-tuned {ft_metrics.box.map50:.3f}')
for i, c in enumerate(ft_metrics.box.ap_class_index):
    print(f'{names[int(c)]:14s} AP50 {ft_metrics.box.ap50[i]:.3f}')

## 7. Export exactly like tools/export_yolo.py and download

Dynamic input shape (the node reshapes to `input_width` x `input_height`), opset 17, plus the
class-name YAML the node reads.

In [ ]:
import yaml
onnx_path = best.export(format='onnx', dynamic=True, simplify=True, opset=17, imgsz=640)
out = '/content/yolo11n_semantic_nav'
shutil.copy(onnx_path, out + '.onnx')
with open(out + '.yaml', 'w') as f:
    yaml.safe_dump({'source_weights': 'fine-tuned ' + BASE,
                    'class_names': [names[i] for i in range(len(names))]}, f, sort_keys=False)
from google.colab import files
files.download(out + '.onnx'); files.download(out + '.yaml')

## 8. Use it on the robot

```
cp ~/Downloads/yolo11n_semantic_nav.{onnx,yaml} ~/semantic_ws/src/semantic_nav_perception/models/
colcon build --symlink-install --packages-select semantic_nav_perception
# params.yaml: model_path: models/yolo11n_semantic_nav.onnx
#              class_names_path: models/yolo11n_semantic_nav.yaml
ros2 run semantic_nav_perception evaluate_detector --data <a NEW capture, not the training one> \
    --model models/yolo11n_semantic_nav.onnx
```
Evaluate on frames that were **not** used for training (record a second tour, or a changed
world) -- otherwise you are measuring memorisation.